# LeetCode 4029 — Elevator Requests IV bằng Python

- Link: https://leetcode.com/problems/elevator-requests-iv/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động theo đoạn (interval DP), Sắp xếp, Bài toán người bán hàng trên đường thẳng, Mô phỏng thời gian chờ

> **Bài Premium:** đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium (đề thực tế lấy từ bản doocs). Chữ ký hàm dưới đây suy ra từ các bài cùng họ (Elevator Requests II/III dùng `elevatorRequests`); nếu LeetCode hiển thị tên khác, chỉ cần đổi tên phương thức.

Skeleton (Python3):

```python
class Solution:
    def elevatorRequests(self, n: int, start: int, requests: list[list[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Một thang máy đứng ở tầng `start` lúc thời điểm 0. Mỗi yêu cầu `[arrival, floor]` chỉ có thể được phục vụ khi thang ở tầng `floor` vào lúc **không sớm hơn** `arrival`. Mỗi giây thang có thể lên 1 tầng, xuống 1 tầng, hoặc đứng yên. Tìm **thời điểm nhỏ nhất** mà mọi yêu cầu đều được phục vụ.

Phiên bản "III" của bài chỉ có `m ≤ 16` yêu cầu nên giải bằng DP tập con (như người bán hàng). Ở bản IV có tới `m ≤ 500` yêu cầu: tập con là bất khả thi, nhưng ta khai thác thực tế là **các tầng nằm trên một đường thẳng**, nhờ đó chỉ cần DP theo **đoạn** `O(m²)`.

Sau notebook này bạn sẽ:

1. Mô hình hoá bài toán: "đến tầng `f` rồi chờ tới `arrival`" ⇔ `t = max(t + khoảng_cách, arrival)`.
2. Viết trọng tài vét cạn: BFS theo từng giây và thử mọi hoán vị thứ tự phục vụ.
3. Hiểu DP tập con (Held–Karp) ở bản III và vì sao nó chết ở `m = 500`.
4. Nhìn ra **cấu trúc "lần ghé cuối"**: thứ tự các lần ghé cuối đi từ **ngoài vào trong** của đoạn tầng, nên trạng thái chỉ cần là một đoạn liên tiếp.
5. Cài DP đoạn `O(m²)` và đối chiếu mọi cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Tòa nhà có `n` tầng đánh số `0..n-1`. Thang bắt đầu ở tầng `start` tại thời điểm `0`.
- Mỗi giây thang **lên 1**, **xuống 1** hoặc **đứng yên**.
- `requests[i] = [arrival_i, floor_i]`: yêu cầu tầng `floor_i` xuất hiện lúc `arrival_i`. Nó được phục vụ **ngay lập tức** nếu thang ở tầng `floor_i` ở **bất kỳ thời điểm nào từ `arrival_i` trở đi**.
- Trả về thời điểm **nhỏ nhất** để phục vụ xong **tất cả** yêu cầu.

**Ví dụ 1:** `n = 9, start = 0, requests = [[0,8],[6,5]]` → `9`. Đi lên tầng 5 (đến lúc 5, đợi tới 6), rồi lên tầng 8 (lúc 9).

**Ví dụ 2:** `n = 8, start = 5, requests = [[1,7],[7,3]]` → `7`. Lên tầng 7 (lúc 2, yêu cầu đã tới nên xong ngay), rồi xuống tầng 3 (đến lúc 6, đợi tới 7).

**Ví dụ 3:** `n = 7, start = 3, requests = [[0,5],[0,1],[6,3]]` → `8`. Tầng 5 (lúc 2), tầng 1 (lúc 6), rồi về tầng 3 (lúc 8, yêu cầu đến lúc 6).

**Ràng buộc**

- `1 <= n <= 10^9`
- `1 <= requests.length <= 500`
- `0 <= arrival_i <= 10^9`
- `0 <= start, floor_i <= n - 1`

Lưu ý: các tầng **không** được nói là khác nhau, nên có thể có nhiều yêu cầu trùng tầng (với thời điểm khác nhau).

In [1]:
EXAMPLES = [
    (9, 0, [[0, 8], [6, 5]], 9),
    (8, 5, [[1, 7], [7, 3]], 7),
    (7, 3, [[0, 5], [0, 1], [6, 3]], 8),
]
for n, start, reqs, exp in EXAMPLES:
    print(f"n={n}, start={start}, requests={reqs} -> kỳ vọng {exp}")

n=9, start=0, requests=[[0, 8], [6, 5]] -> kỳ vọng 9
n=8, start=5, requests=[[1, 7], [7, 3]] -> kỳ vọng 7
n=7, start=3, requests=[[0, 5], [0, 1], [6, 3]] -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1. Khoảng cách trên một đường thẳng: `abs`

Thang đi từ tầng `a` đến tầng `b` mất đúng `abs(a - b)` giây (đi thẳng, không vòng).

In [2]:
print(abs(5 - 0), abs(0 - 5), abs(3 - 3))
# Đi 0 -> 5 -> 8 tốn:
print(abs(0 - 5) + abs(5 - 8))

5 5 0
8


### 3.2. "Đến sớm thì chờ" = `max(thời_điểm_đến, arrival)`

Nếu thang rời một tầng ở thời điểm `t`, đến tầng `f` lúc `t + d`. Yêu cầu chỉ phục vụ được từ `arrival` trở đi, nên thời điểm phục vụ là `max(t + d, arrival)` — đến sớm thì **đứng chờ** (được phép vì thang có thể đứng yên), đến muộn thì phục vụ ngay.

In [3]:
def serve_time(t, pos, floor, arrival):
    return max(t + abs(pos - floor), arrival)

print(serve_time(0, 0, 5, 6))   # đến lúc 5 nhưng yêu cầu lúc 6 -> 6
print(serve_time(6, 5, 8, 0))   # 6 + 3 = 9, yêu cầu đã có -> 9

6
9


### 3.3. Mô phỏng một thứ tự phục vụ cho trước

Nếu ta cố định **thứ tự** các yêu cầu được phục vụ, thời điểm kết thúc tính bằng một vòng lặp áp dụng `serve_time` liên tiếp. Việc còn lại là chọn thứ tự tốt nhất.

In [4]:
def finish_time(start, order):
    """order: list các (arrival, floor) theo thứ tự phục vụ."""
    t, pos = 0, start
    for arrival, floor in order:
        t = max(t + abs(pos - floor), arrival)
        pos = floor
    return t

# Ví dụ 3: thứ tự 5 -> 1 -> 3
print(finish_time(3, [(0, 5), (0, 1), (6, 3)]))      # 8
# Thứ tự khác: 1 -> 5 -> 3
print(finish_time(3, [(0, 1), (0, 5), (6, 3)]))      # 3->1 (2), 1->5 (4) => 6, rồi 5->3: max(6+2, 6) = 8

8
8


### 3.4. `sorted` với khoá, `itertools.permutations`, và bitmask

- `sorted(requests, key=lambda r: r[1])` sắp theo tầng (phần tử thứ hai).
- `itertools.permutations(items)` sinh mọi hoán vị — `m!` thứ tự, chỉ dùng cho `m ≤ 7`.
- Bitmask: `mask & (1 << k)` kiểm tra yêu cầu `k` đã xong chưa; `mask | (1 << k)` thêm nó vào tập.

In [5]:
from itertools import permutations

reqs = [[6, 5], [0, 8], [0, 1]]
print(sorted(reqs, key=lambda r: r[1]))              # theo tầng
print(len(list(permutations(range(4)))), "hoán vị của 4 phần tử")
mask = 0
for k in (0, 2):
    mask |= 1 << k
print(bin(mask), [bool(mask & (1 << k)) for k in range(3)])

[[0, 1], [6, 5], [0, 8]]
24 hoán vị của 4 phần tử
0b101 [True, False, True]


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 1** (`start = 0`, yêu cầu `(arrival 0, tầng 8)` và `(arrival 6, tầng 5)`):

- Thứ tự "8 rồi 5": đến tầng 8 lúc 8, rồi xuống tầng 5 lúc 11 → **11**.
- Thứ tự "5 rồi 8": đến tầng 5 lúc 5, chờ tới 6; lên tầng 8 lúc 9 → **9**.

Vậy chọn 9. Hai điều rút ra:

1. Đáp án phụ thuộc **thứ tự** ghé các tầng và việc **chờ**.
2. Nhìn lướt thì giống bài người bán hàng (TSP): chọn hoán vị tốt nhất. Với `m = 16` ta dùng DP tập con. Với `m = 500` phải khai thác **đường thẳng**.

## 5. Cách 1 — Vét cạn: mô phỏng từng giây (BFS theo thời gian) và thử mọi hoán vị

**Cách 1a — BFS theo từng giây (nguyên văn đề).** Trạng thái `(tầng, mask các yêu cầu đã xong)`. Ở mỗi giây, từ mỗi trạng thái ta đi lên / xuống / đứng yên (tầng phải nằm trong `[0, n-1]`), rồi đánh dấu mọi yêu cầu đang ở tầng hiện tại và đã tới `arrival`. Dừng ở giây đầu tiên có trạng thái "mask đầy đủ". Chỉ dùng được với `n`, `arrival` nhỏ.

**Cách 1b — Thử mọi hoán vị.** Với mỗi thứ tự, dùng `finish_time` ở 3.3. `O(m! · m)`.

In [6]:
def elevator_bfs(n, start, requests, return_home=False):
    m = len(requests)
    full = (1 << m) - 1

    def fulfil(t, floor, mask):
        for k, (arrival, f) in enumerate(requests):
            if f == floor and t >= arrival:
                mask |= 1 << k
        return mask

    cur = {(start, fulfil(0, start, 0))}
    t = 0
    while True:
        for floor, mask in cur:
            if mask == full and (not return_home or floor == start):
                return t
        t += 1
        nxt = set()
        for floor, mask in cur:
            for d in (-1, 0, 1):
                nf = floor + d
                if 0 <= nf < n:
                    nxt.add((nf, fulfil(t, nf, mask)))
        cur = nxt

def elevator_permutations(n, start, requests, return_home=False):
    best = float("inf")
    for order in permutations(requests):
        t = finish_time(start, order)
        if return_home:
            t += abs(order[-1][1] - start)
        best = min(best, t)
    return best

for n, start, reqs, exp in EXAMPLES:
    print(elevator_bfs(n, start, reqs), elevator_permutations(n, start, reqs), exp)

9 9 9
7 7 7
8 8 8


**Độ phức tạp.** BFS: `O(T · n · 2^m)` với `T` là đáp án — chỉ cho thang rất nhỏ. Hoán vị: `O(m! · m)`.

**Nút thắt:** `m!` bùng nổ. Cùng một "tập đã phục vụ + tầng cuối" xuất hiện trong rất nhiều hoán vị, và với cùng trạng thái đó **thời điểm nhỏ hơn luôn tốt hơn** (vì có thể chờ) — đó là gợi ý DP.

## 6. Bước đột phá

### 6.1. Thời điểm sớm hơn luôn không tệ hơn

Nếu hai kế hoạch cùng phục vụ một tập yêu cầu và cùng kết thúc ở một tầng, kế hoạch kết thúc **sớm hơn** có thể bắt chước kế hoạch kia bằng cách đứng chờ. Vì vậy với mỗi trạng thái ta chỉ cần giữ **thời điểm nhỏ nhất**.

### 6.2. Cấu trúc "lần ghé cuối" trên đường thẳng

Xét một lộ trình tối ưu và với mỗi yêu cầu `k`, gọi `L_k` là **lần cuối** thang ở tầng `floor_k`. Yêu cầu `k` được phục vụ khi và chỉ khi `L_k ≥ arrival_k` (vì ở lần cuối, thang đang ở tầng đó muộn nhất có thể). Do đó ta chỉ quan tâm các `L_k`.

Tại thời điểm `t`, tập các tầng mà thang **sẽ còn ghé từ `t` trở đi** là một **đoạn liên tục** (đường đi liên tục trên đường thẳng nên lấp đầy mọi tầng giữa các điểm nó chạm). Khi `t` tăng, đoạn này **chỉ co lại**. Các yêu cầu bị "rơi ra" khỏi đoạn tại thời điểm `L_k` — luôn là yêu cầu ở **đầu trái** hoặc **đầu phải** của đoạn.

Hệ quả: sắp các yêu cầu theo tầng, thứ tự các lần ghé cuối là: lần lượt **bóc một yêu cầu từ đầu trái hoặc đầu phải** của danh sách còn lại, tới khi hết. Trạng thái chỉ cần: **đã bóc bao nhiêu ở trái (`l`) và ở phải (`r`)**, và thang đang ở yêu cầu vừa bóc ở phía nào.

### 6.3. Công thức chuyển

Sau khi sắp theo tầng `F[0..m-1]`, `A[0..m-1]` (arrival tương ứng). Trạng thái `(l, r, side)`:

- `side = 0`: thang ở tầng `F[l-1]` (vừa bóc yêu cầu trái thứ `l-1`); `side = 1`: ở tầng `F[m-r]`.
- `dp[l][r][side]` = thời điểm nhỏ nhất của trạng thái đó.
- Bóc trái: `dp[l+1][r][0] = max(t + |pos - F[l]|, A[l])`.
- Bóc phải: `dp[l][r+1][1] = max(t + |pos - F[m-1-r]|, A[m-1-r])`.
- Khởi đầu: thang ở `start`, `t = 0`, bóc trái đầu tiên hoặc bóc phải đầu tiên.
- Đáp án: min `dp[l][r][side]` với `l + r = m`.

**Quy tắc nếu… thì…:** mỗi bước chọn bóc đầu trái hoặc đầu phải; thời điểm phục vụ luôn là `max(đến nơi, arrival)`.

**Bất biến.** `dp[l][r][side]` là thời điểm sớm nhất mà ở đó `l` yêu cầu trái nhất và `r` yêu cầu phải nhất đều đã có "lần ghé cuối" (mọi lần ghé cuối của phần còn lại sẽ xảy ra sau).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP tập con (Held–Karp), bản dành cho `m ≤ 16`

`f[mask][j]` = thời điểm nhỏ nhất để đã phục vụ tập `mask`, yêu cầu **cuối cùng** là `j`.

- `mask` chỉ chứa `j`: `f[mask][j] = max(|start - floor_j|, arrival_j)`.
- Ngược lại: `f[mask][j] = min over j0 in mask\{j} của max(f[mask\{j}][j0] + |floor_j0 - floor_j|, arrival_j)`.
- Đáp án: `min_j f[full][j]`.

Giải thích: đây là bản "TSP có cửa sổ thời gian bắt đầu". Mốc thời gian nhỏ nhất luôn tốt nhất theo 6.1.

In [7]:
def elevator_bitmask(n, start, requests):
    m = len(requests)
    INF = float("inf")
    f = [[INF] * m for _ in range(1 << m)]
    for j, (a, fl) in enumerate(requests):
        f[1 << j][j] = max(abs(start - fl), a)
    for mask in range(1, 1 << m):
        for j in range(m):
            t = f[mask][j]
            if t == INF:
                continue
            for k in range(m):
                if mask & (1 << k):
                    continue
                a, fl = requests[k]
                nt = max(t + abs(requests[j][1] - fl), a)
                if nt < f[mask | (1 << k)][k]:
                    f[mask | (1 << k)][k] = nt
    return min(f[(1 << m) - 1])

for n, start, reqs, exp in EXAMPLES:
    print(elevator_bitmask(n, start, reqs), exp)

9 9
7 7
8 8


**Độ phức tạp.** `O(m² · 2^m)` thời gian, `O(m · 2^m)` bộ nhớ. Với `m = 16` ổn (≈ 17M thao tác), với `m = 500` thì không thể (`2^500`).

### 7.2. Cách 3 — DP đoạn `O(m²)` (bản nộp)

**Giải thích từng đoạn.**

1. Sắp `requests` theo tầng (nếu trùng tầng, thứ tự tuỳ ý): `F` (tầng), `A` (arrival).
2. `dp[l][r][side]` khởi tạo vô cực. Đặt trạng thái khởi đầu từ `start`: bóc trái đầu tiên (`l=1, r=0, side=0`) hoặc bóc phải đầu tiên (`l=0, r=1, side=1`).
3. Duyệt theo `d = l + r = 1..m`. Với mỗi `(l, r, side)` hợp lệ: nếu `l + r == m` thì cập nhật đáp án; ngược lại thử bóc trái hoặc bóc phải.
4. Vị trí hiện tại: `side == 0` ⇒ `F[l-1]`; `side == 1` ⇒ `F[m - r]`.

In [8]:
def elevator_interval(n, start, requests, return_home=False):
    R = sorted((f, a) for a, f in requests)         # sắp theo tầng
    m = len(R)
    F = [x[0] for x in R]
    A = [x[1] for x in R]
    INF = float("inf")
    dp = [[[INF, INF] for _ in range(m + 1)] for _ in range(m + 1)]

    dp[1][0][0] = max(abs(start - F[0]), A[0])               # bóc trái đầu tiên
    dp[0][1][1] = min(dp[0][1][1], max(abs(start - F[m - 1]), A[m - 1]))   # bóc phải đầu tiên

    best = INF
    for d in range(1, m + 1):
        for l in range(d + 1):
            r = d - l
            for side in (0, 1):
                t = dp[l][r][side]
                if t == INF:
                    continue
                pos = F[l - 1] if side == 0 else F[m - r]
                if l + r == m:
                    best = min(best, t + (abs(pos - start) if return_home else 0))
                    continue
                nt = max(t + abs(pos - F[l]), A[l])                    # bóc trái
                if nt < dp[l + 1][r][0]:
                    dp[l + 1][r][0] = nt
                nt = max(t + abs(pos - F[m - 1 - r]), A[m - 1 - r])    # bóc phải
                if nt < dp[l][r + 1][1]:
                    dp[l][r + 1][1] = nt
    return best

for n, start, reqs, exp in EXAMPLES:
    print(elevator_interval(n, start, reqs), exp)

9 9
7 7
8 8


**Độ phức tạp.** `O(m²)` trạng thái, mỗi trạng thái `O(1)` chuyển ⇒ `O(m²)` thời gian, `O(m²)` bộ nhớ (có thể giảm xuống `O(m)` bằng cách chỉ giữ đường chéo `l + r = d`, bản nộp giữ nguyên bảng cho dễ đọc). Với `m = 500`: khoảng `500 · 500 · 2 = 5·10^5` trạng thái.

Chú ý: `n` **không** xuất hiện trong độ phức tạp — chỉ khoảng cách giữa các tầng yêu cầu quan trọng.

### 7.3. Cách 4 — Cùng DP đoạn nhưng cuộn theo đường chéo (tiết kiệm bộ nhớ)

Trạng thái ở đường chéo `d` chỉ phụ thuộc đường chéo `d - 1`. Ta giữ hai mảng `cur[l][side]` theo `l` (vì `r = d - l`). Bộ nhớ `O(m)`.

In [9]:
def elevator_interval_rolling(n, start, requests):
    R = sorted((f, a) for a, f in requests)
    m = len(R)
    F = [x[0] for x in R]
    A = [x[1] for x in R]
    INF = float("inf")
    # cur[l][side] với d = l + r
    cur = [[INF, INF] for _ in range(m + 2)]
    cur[1][0] = max(abs(start - F[0]), A[0])                       # d = 1: l=1,r=0
    cur[0][1] = min(cur[0][1], max(abs(start - F[m - 1]), A[m - 1]))   # d = 1: l=0,r=1
    for d in range(1, m):
        nxt = [[INF, INF] for _ in range(m + 2)]
        for l in range(d + 1):
            r = d - l
            for side in (0, 1):
                t = cur[l][side]
                if t == INF:
                    continue
                pos = F[l - 1] if side == 0 else F[m - r]
                nt = max(t + abs(pos - F[l]), A[l])
                if nt < nxt[l + 1][0]:
                    nxt[l + 1][0] = nt
                nt = max(t + abs(pos - F[m - 1 - r]), A[m - 1 - r])
                if nt < nxt[l][1]:
                    nxt[l][1] = nt
        cur = nxt
    return min(min(row) for row in cur)

for n, start, reqs, exp in EXAMPLES:
    print(elevator_interval_rolling(n, start, reqs), exp)

9 9
7 7
8 8


## 8. TRACE từng bước

Trace Cách 3 trên **ví dụ 3**: `start = 3`, `requests = [[0,5],[0,1],[6,3]]`. Sắp theo tầng: `F = [1, 3, 5]`, `A = [0, 6, 0]`.

| d | (l, r, side) | thang ở | t | hành động tiếp |
|---|---|---|---|---|
| 1 | (1,0,0) | tầng 1 | `max(|3-1|, 0) = 2` | bóc trái tiếp tầng 3 hoặc bóc phải tầng 5 |
| 1 | (0,1,1) | tầng 5 | `max(|3-5|, 0) = 2` | bóc trái tầng 1 hoặc bóc phải tầng 3 |
| 2 | (0,2,1) | tầng 3 | từ (0,1,1): `max(2 + 2, 6) = 6` | bóc nốt tầng 1 |
| 2 | (1,1,0) | tầng 1 | từ (0,1,1): `max(2 + 4, 0) = 6` | bóc nốt tầng 3 |
| 3 | (1,2,0) | tầng 1 | từ (0,2,1) bóc trái tầng 1: `max(6 + 2, 0) = 8` | kết thúc, đáp án 8 |
| 3 | (2,1,0) | tầng 3 | từ (1,1,0) bóc trái tầng 3: `max(6 + 2, 6) = 8` | kết thúc, đáp án 8 |

Các nhánh còn lại (bóc tầng 1 trước, tầng 3 sau) cho ≥ 8. Đáp án `8`.

In [10]:
def trace_elevator(n, start, requests):
    R = sorted((f, a) for a, f in requests)
    m = len(R)
    F = [x[0] for x in R]; A = [x[1] for x in R]
    INF = float("inf")
    print("tầng đã sắp F =", F, "| arrival A =", A)
    dp = {}
    def relax(l, r, side, t, why):
        key = (l, r, side)
        if t < dp.get(key, INF):
            dp[key] = t
            print(f"   dp[l={l}, r={r}, side={side}] = {t}   ({why})")
    relax(1, 0, 0, max(abs(start - F[0]), A[0]), f"start {start} -> tầng {F[0]}, chờ tới {A[0]}")
    relax(0, 1, 1, max(abs(start - F[m - 1]), A[m - 1]), f"start {start} -> tầng {F[m-1]}, chờ tới {A[m-1]}")
    best = INF
    for d in range(1, m + 1):
        print(f"-- đường chéo d = {d}")
        for l in range(d + 1):
            r = d - l
            for side in (0, 1):
                t = dp.get((l, r, side), INF)
                if t == INF:
                    continue
                pos = F[l - 1] if side == 0 else F[m - r]
                if l + r == m:
                    best = min(best, t)
                    print(f"   kết thúc ở (l={l}, r={r}, side={side}) lúc {t}")
                    continue
                relax(l + 1, r, 0, max(t + abs(pos - F[l]), A[l]), f"từ tầng {pos} bóc trái tầng {F[l]} (A={A[l]})")
                relax(l, r + 1, 1, max(t + abs(pos - F[m - 1 - r]), A[m - 1 - r]), f"từ tầng {pos} bóc phải tầng {F[m-1-r]} (A={A[m-1-r]})")
    print("Đáp án =", best)
    return best

trace_elevator(*EXAMPLES[2][:3])

tầng đã sắp F = [1, 3, 5] | arrival A = [0, 6, 0]
   dp[l=1, r=0, side=0] = 2   (start 3 -> tầng 1, chờ tới 0)
   dp[l=0, r=1, side=1] = 2   (start 3 -> tầng 5, chờ tới 0)
-- đường chéo d = 1
   dp[l=1, r=1, side=0] = 6   (từ tầng 5 bóc trái tầng 1 (A=0))
   dp[l=0, r=2, side=1] = 6   (từ tầng 5 bóc phải tầng 3 (A=6))
   dp[l=2, r=0, side=0] = 6   (từ tầng 1 bóc trái tầng 3 (A=6))
   dp[l=1, r=1, side=1] = 6   (từ tầng 1 bóc phải tầng 5 (A=0))
-- đường chéo d = 2
   dp[l=1, r=2, side=0] = 8   (từ tầng 3 bóc trái tầng 1 (A=0))
   dp[l=0, r=3, side=1] = 8   (từ tầng 3 bóc phải tầng 1 (A=0))
   dp[l=2, r=1, side=0] = 8   (từ tầng 1 bóc trái tầng 3 (A=6))
   dp[l=1, r=2, side=1] = 8   (từ tầng 1 bóc phải tầng 3 (A=6))
   dp[l=3, r=0, side=0] = 8   (từ tầng 3 bóc trái tầng 5 (A=0))
   dp[l=2, r=1, side=1] = 8   (từ tầng 3 bóc phải tầng 5 (A=0))
-- đường chéo d = 3
   kết thúc ở (l=0, r=3, side=1) lúc 8
   kết thúc ở (l=1, r=2, side=0) lúc 8
   kết thúc ở (l=1, r=2, side=1) lúc 8
   kết thúc

8

### 8.2. Trace ví dụ 1 (`start = 0`, `[[0,8],[6,5]]`)

Sắp theo tầng: `F = [5, 8]`, `A = [6, 0]`. Bóc trái (tầng 5) trước: `t = max(5, 6) = 6`; rồi bóc tầng 8: `t = max(6 + 3, 0) = 9`. Bóc phải (tầng 8) trước: `t = 8`, rồi tầng 5: `max(8 + 3, 6) = 11`. Chọn 9.

In [11]:
trace_elevator(*EXAMPLES[0][:3])

tầng đã sắp F = [5, 8] | arrival A = [6, 0]
   dp[l=1, r=0, side=0] = 6   (start 0 -> tầng 5, chờ tới 6)
   dp[l=0, r=1, side=1] = 8   (start 0 -> tầng 8, chờ tới 0)
-- đường chéo d = 1
   dp[l=1, r=1, side=0] = 11   (từ tầng 8 bóc trái tầng 5 (A=6))
   dp[l=0, r=2, side=1] = 11   (từ tầng 8 bóc phải tầng 5 (A=6))
   dp[l=2, r=0, side=0] = 9   (từ tầng 5 bóc trái tầng 8 (A=0))
   dp[l=1, r=1, side=1] = 9   (từ tầng 5 bóc phải tầng 8 (A=0))
-- đường chéo d = 2
   kết thúc ở (l=0, r=2, side=1) lúc 11
   kết thúc ở (l=1, r=1, side=0) lúc 11
   kết thúc ở (l=1, r=1, side=1) lúc 9
   kết thúc ở (l=2, r=0, side=0) lúc 9
Đáp án = 9


9

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (DP đoạn). Ta giữ tên phương thức `elevatorRequests` như các bài cùng họ.

In [12]:
class Solution:
    def elevatorRequests(self, n: int, start: int, requests: list[list[int]]) -> int:
        # Sắp theo tầng: F[i] tầng, A[i] thời điểm yêu cầu xuất hiện
        reqs = sorted((f, a) for a, f in requests)
        m = len(reqs)
        F = [x[0] for x in reqs]
        A = [x[1] for x in reqs]
        INF = float("inf")

        # dp[l][r][side]: đã "bóc" l yêu cầu từ trái, r yêu cầu từ phải;
        # side 0: thang ở tầng F[l-1]; side 1: thang ở tầng F[m-r]. Giá trị: thời điểm nhỏ nhất.
        dp = [[[INF, INF] for _ in range(m + 1)] for _ in range(m + 1)]
        dp[1][0][0] = max(abs(start - F[0]), A[0])
        if dp[0][1][1] > max(abs(start - F[m - 1]), A[m - 1]):
            dp[0][1][1] = max(abs(start - F[m - 1]), A[m - 1])

        ans = INF
        for d in range(1, m + 1):
            for l in range(d + 1):
                r = d - l
                for side in (0, 1):
                    t = dp[l][r][side]
                    if t == INF:
                        continue
                    pos = F[l - 1] if side == 0 else F[m - r]
                    if d == m:
                        ans = min(ans, t)
                        continue
                    # bóc đầu trái
                    nt = max(t + abs(pos - F[l]), A[l])
                    if nt < dp[l + 1][r][0]:
                        dp[l + 1][r][0] = nt
                    # bóc đầu phải
                    j = m - 1 - r
                    nt = max(t + abs(pos - F[j]), A[j])
                    if nt < dp[l][r + 1][1]:
                        dp[l][r + 1][1] = nt
        return ans


sol = Solution()
for n, start, reqs, exp in EXAMPLES:
    assert sol.elevatorRequests(n, start, reqs) == exp, (n, start, reqs)
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) DP chỉ tạo lộ trình hợp lệ ⇒ không thấp hơn đáp án thật.** Mỗi chuỗi bóc tương ứng với một thứ tự phục vụ; thời điểm `max(t + khoảng_cách, arrival)` chính là thời điểm của lộ trình "đi thẳng tới tầng tiếp theo, đợi nếu cần". Đó là một lộ trình thật nên đáp án DP `≥` tối ưu.

**(b) Tối ưu thật luôn biểu diễn được ⇒ DP không cao hơn.** Lấy lộ trình tối ưu và các lần ghé cuối `L_k`. Theo 6.2, tập tầng còn ghé trong tương lai là đoạn co dần nên thứ tự `L_k` tăng dần tương ứng một chuỗi bóc từ hai đầu. Khi ta thay lộ trình thật bằng "đi thẳng giữa các tầng theo thứ tự bóc", thời điểm tại mỗi bước không lớn hơn thời điểm thật: bằng quy nạp, `t_DP(k) = max(t_DP(k-1) + d, A_k) ≤ max(L_{k-1} + d, A_k) ≤ L_k` vì lộ trình thật phải đi ít nhất `d` giữa hai lần ghé cuối và `L_k ≥ A_k`.

**(c) Giữ thời điểm nhỏ nhất đủ.** Thời điểm sớm hơn ở cùng trạng thái luôn không tệ hơn (có thể chờ), nên `min` là đúng.

**(d) Yêu cầu trùng tầng.** Chúng nằm kề nhau trong danh sách sắp xếp; khoảng cách giữa chúng 0, nên bóc chúng theo thứ tự nào cũng cho cùng thời điểm `max(..., A)`.

**(e) Ví dụ minh hoạ.** Ở ví dụ 1, thứ tự lần ghé cuối là "tầng 5 rồi tầng 8" (bóc trái rồi bóc phải) ⇒ 9; thứ tự ngược cho 11.

## 11. Phân tích Time/Space

Đặt `m = len(requests)` (≤ 500).

| Cách | Time | Space |
|---|---|---|
| 1a. BFS từng giây | `O(T · n · 2^m)` | `O(n · 2^m)` |
| 1b. Hoán vị | `O(m! · m)` | `O(m)` |
| 2. DP tập con | `O(m² · 2^m)` | `O(m · 2^m)` |
| 3. DP đoạn | `O(m²)` | `O(m²)` |
| 4. DP đoạn cuộn đường chéo | `O(m²)` | `O(m)` |

`n` (tới `10^9`) **không** ảnh hưởng thời gian chạy, chỉ giá trị khoảng cách (số nguyên Python không tràn).

In [13]:
import random, time

rng = random.Random(4029)
m = 500
reqs = [[rng.randint(0, 10**9), rng.randint(0, 10**9 - 1)] for _ in range(m)]
t = time.time()
ans = Solution().elevatorRequests(10**9, rng.randint(0, 10**9 - 1), reqs)
print("m=500 ngẫu nhiên:", ans, f"({time.time() - t:.2f}s)")
t = time.time()
ans2 = elevator_interval_rolling(10**9, 0, reqs)
print("bản cuộn:", ans2, f"({time.time() - t:.2f}s)")

m=500 ngẫu nhiên: 1935925543 (0.14s)


bản cuộn: 1935925543 (0.08s)


## 12. Bộ test

Ta kiểm: ví dụ đề, edge case, rồi đối chiếu **BFS từng giây** (nguyên văn đề), hoán vị, DP tập con, DP đoạn, bản cuộn và `Solution` trên input ngẫu nhiên nhỏ có seed — gồm cả yêu cầu trùng tầng.

In [14]:
ALL_SMALL = [
    lambda n, s, r: elevator_bfs(n, s, r),
    lambda n, s, r: elevator_permutations(n, s, r),
    lambda n, s, r: elevator_bitmask(n, s, r),
    lambda n, s, r: elevator_interval(n, s, r),
    lambda n, s, r: elevator_interval_rolling(n, s, r),
    lambda n, s, r: Solution().elevatorRequests(n, s, r),
]
EDGE = [
    (1, 0, [[0, 0]], 0),                      # đã ở đúng tầng, arrival 0
    (1, 0, [[5, 0]], 5),                      # đứng yên chờ tới 5
    (5, 2, [[0, 2], [0, 2]], 0),              # hai yêu cầu cùng tầng
    (5, 0, [[0, 4]], 4),
    (5, 4, [[0, 0]], 4),
    (6, 0, [[0, 3], [10, 3]], 10),            # cùng tầng khác arrival
    (6, 3, [[0, 0], [0, 5]], 7),              # 3 -> 5 (2 giây) rồi 5 -> 0 (5 giây)
    (10, 4, [[0, 0], [0, 9], [20, 4]], 20),   # bị chặn bởi arrival 20
]
for n, s, r, exp in EDGE + EXAMPLES:
    for fn in ALL_SMALL:
        assert fn(n, s, r) == exp, (n, s, r, fn(n, s, r), exp)
print("Ví dụ đề + edge case: mọi cách đúng")

rng = random.Random(402901)
for _ in range(500):
    m = rng.randint(1, 4)
    n = rng.randint(1, 8)
    reqs = [[rng.randint(0, 8), rng.randint(0, n - 1)] for _ in range(m)]
    s = rng.randint(0, n - 1)
    vals = [fn(n, s, reqs) for fn in ALL_SMALL]
    assert len(set(vals)) == 1, (n, s, reqs, vals)
print("500 input nhỏ ngẫu nhiên: 6 cách khớp nhau (BFS là trọng tài nguyên văn đề)")

Ví dụ đề + edge case: mọi cách đúng
500 input nhỏ ngẫu nhiên: 6 cách khớp nhau (BFS là trọng tài nguyên văn đề)


In [15]:
# Input lớn hơn: các cách không-BFS khớp nhau
rng = random.Random(7)
for _ in range(300):
    m = rng.randint(1, 8)
    n = rng.randint(1, 40)
    reqs = [[rng.randint(0, 60), rng.randint(0, n - 1)] for _ in range(m)]
    s = rng.randint(0, n - 1)
    a = elevator_permutations(n, s, reqs)
    assert a == elevator_bitmask(n, s, reqs) == elevator_interval(n, s, reqs) == elevator_interval_rolling(n, s, reqs) == Solution().elevatorRequests(n, s, reqs), (n, s, reqs)
print("300 input vừa (m ≤ 8, arrival ≤ 60): các cách khớp hoán vị")

# Input không bị thay đổi
reqs = [[0, 8], [6, 5]]
snap = [r[:] for r in reqs]
Solution().elevatorRequests(9, 0, reqs)
assert reqs == snap
print("requests không bị thay đổi")

300 input vừa (m ≤ 8, arrival ≤ 60): các cách khớp hoán vị
requests không bị thay đổi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Phục vụ theo thứ tự `arrival` (hoặc tham lam "tầng gần nhất")

**Code sai:** sắp theo `arrival` rồi đi lần lượt.
**Vì sao sai:** ví dụ 2 (`start = 5`, `[[1,7],[7,3]]`) cho kết quả đúng, nhưng ví dụ 1 (`[[0,8],[6,5]]`) lại sai: yêu cầu xuất hiện trước (tầng 8) chưa chắc nên làm trước, vì đi qua tầng 5 trên đường có thể chờ tới lúc yêu cầu của nó.
**Code đúng:** DP đoạn / DP tập con tìm thứ tự tốt nhất.

In [16]:
def wrong_by_arrival(n, start, requests):
    order = sorted((a, f) for a, f in requests)
    return finish_time(start, order)

def wrong_nearest_first(n, start, requests):
    left, pos, t = list(requests), start, 0
    while left:
        left.sort(key=lambda r: (abs(pos - r[1]), r[0]))
        a, f = left.pop(0)
        t = max(t + abs(pos - f), a); pos = f
    return t

print("theo arrival:", wrong_by_arrival(9, 0, [[0, 8], [6, 5]]), "| đúng: 9")
print("gần nhất:", wrong_nearest_first(7, 3, [[0, 5], [0, 1], [6, 3]]), "| đúng: 8")
rng = random.Random(3)
shown = 0
for _ in range(5000):
    n = rng.randint(3, 12)
    reqs = [[rng.randint(0, 12), rng.randint(0, n - 1)] for _ in range(rng.randint(2, 5))]
    s = rng.randint(0, n - 1)
    if wrong_nearest_first(n, s, reqs) != Solution().elevatorRequests(n, s, reqs):
        print("phản ví dụ cho 'gần nhất':", n, s, reqs, wrong_nearest_first(n, s, reqs), "!=", Solution().elevatorRequests(n, s, reqs))
        shown += 1
        if shown >= 2:
            break

theo arrival: 11 | đúng: 9
gần nhất: 12 | đúng: 8
phản ví dụ cho 'gần nhất': 3 2 [[4, 2], [3, 0], [11, 1], [8, 2], [7, 1]] 12 != 11
phản ví dụ cho 'gần nhất': 5 0 [[10, 1], [8, 3], [11, 0]] 14 != 11


### Lỗi 2 — Quên chờ: `t = t + d` thay vì `max(t + d, arrival)`

**Code sai:** `t += abs(pos - f)`.
**Vì sao sai:** yêu cầu chưa xuất hiện thì chưa phục vụ được; thang phải chờ.
**Code đúng:** `t = max(t + abs(pos - f), arrival)`.

### Lỗi 3 — Quên thang có thể **không** đi thẳng: cộng khoảng cách thay vì lấy `abs`

**Code sai:** `F[l] - pos` (có thể âm).
**Code đúng:** `abs(pos - F[l])`.

### Lỗi 4 — Dùng `dp` ban đầu bằng 0 hoặc quên khởi tạo từ `start`

**Code sai:** `dp[0][0][*] = 0` rồi bóc như thể thang đã ở đầu danh sách.
**Vì sao sai:** thang bắt đầu ở `start`, không phải ở tầng của yêu cầu nào.
**Code đúng:** tính bước đầu `max(abs(start - F[0]), A[0])` và `max(abs(start - F[m-1]), A[m-1])`.

In [17]:
def wrong_no_wait(n, start, requests):
    R = sorted((f, a) for a, f in requests)
    best, F = float("inf"), [x[0] for x in R]
    # bản đơn giản: thử mọi hoán vị nhưng quên chờ
    for order in permutations(requests):
        t, pos = 0, start
        for a, f in order:
            t += abs(pos - f)            # SAI: không chờ arrival
            pos = f
        best = min(best, t)
    return best

print("không chờ:", wrong_no_wait(9, 0, [[0, 8], [6, 5]]), "| đúng: 9")

không chờ: 8 | đúng: 9


## 14. Mẹo & nhận dạng pattern

- **"Ghé các điểm trên đường thẳng, có thể chờ"** → các điểm đã ghé tạo thành đoạn liên tục ⇒ DP theo đoạn `O(m²)`; TSP tổng quát thì cần bitmask.
- **"Cửa sổ thời gian chỉ có cận dưới (release time)"** → bước chuyển luôn là `max(đến nơi, mốc)`; thời điểm sớm hơn không bao giờ tệ.
- **Nghĩ ngược thời gian / lần ghé cuối** biến "phục vụ khi ghé muộn nhất" thành cấu trúc co dần ⇒ bóc từ hai đầu.
- Khi `n` tới `10^9` nhưng `m` nhỏ: chỉ quan trọng các toạ độ yêu cầu, nén về `m` điểm.
- Dùng bitmask DP (bản III) làm **trọng tài** để kiểm tra DP đoạn.
- Với đề không nói "tầng khác nhau", luôn test trường hợp trùng tầng.

## 15. Biến thể

1. **Phải về lại tầng `start` sau khi xong:** đáp án = `min(t + |pos - start|)` ở trạng thái kết thúc (Bài 5; đã đối chiếu với BFS yêu cầu thang về `start`).
2. **Elevator Requests III (m ≤ 16):** DP tập con (Cách 2).
3. **Elevator Requests II:** tổng penalty (tổng thời điểm phục vụ) thay vì thời điểm kết thúc, tất cả `arrival = 0`.
4. **Mỗi yêu cầu có hạn chót (deadline):** thêm điều kiện `t ≤ deadline`; vẫn DP đoạn nhưng cần kiểm tra khả thi theo thứ tự thời gian xuôi.
5. **Thang chở được tối đa `C` người / đi qua nhiều toà nhà:** TSP tổng quát, thường NP-khó.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. BFS từng giây | Mô phỏng nguyên văn | `O(T·n·2^m)` | `O(n·2^m)` | Trọng tài, thang rất nhỏ |
| 1b. Hoán vị | Thử mọi thứ tự | `O(m!·m)` | `O(m)` | `m ≤ 7` |
| 2. DP tập con | Held–Karp | `O(m²·2^m)` | `O(m·2^m)` | `m ≤ 16` (bản III) |
| 3. DP đoạn | Bóc hai đầu | `O(m²)` | `O(m²)` | **Bản nộp**, `m ≤ 500` |
| 4. DP đoạn cuộn | Giữ một đường chéo | `O(m²)` | `O(m)` | Khi `m` rất lớn |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `finish_time(2, [(0, 5), (4, 1)])` bằng bao nhiêu?

**Bài 2 (tính tay).** `n = 10`, `start = 4`, `requests = [[0,0],[0,9],[20,4]]`. Tính các thứ tự "ghé cuối" và cho đáp án. Vì sao yêu cầu `[20, 4]` quyết định kết quả?

**Bài 3 (điền TODO).** Hoàn thiện `step(t, pos, floor, arrival)` trả về thời điểm phục vụ.

**Bài 4 (tự viết lại).** Viết `my_elevator(n, start, requests)` theo DP đoạn.

**Bài 5 (biến thể).** Viết `elevator_round_trip(n, start, requests)`: thang phải **quay về `start`** sau khi phục vụ xong mọi yêu cầu; trả về thời điểm nhỏ nhất thang trở lại `start`.

In [18]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def step(t, pos, floor, arrival):
    # TODO: thời điểm phục vụ khi rời pos lúc t, đến floor, yêu cầu xuất hiện lúc arrival
    pass

# Bài 4 — TODO
def my_elevator(n, start, requests):
    # TODO: sắp theo tầng; dp[l][r][side]; bóc trái / bóc phải; lấy min khi l + r == m
    pass

# Bài 5 — TODO
def elevator_round_trip(n, start, requests):
    # TODO: giống bài 4, nhưng khi kết thúc cộng thêm abs(pos - start)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: đi từ tầng 2 lên tầng 5 mất 3 giây.
- Mức 2: `t = max(0 + 3, 0) = 3`; rồi từ tầng 5 xuống tầng 1 mất 4 giây: `max(3 + 4, 4)`.
- Mức 3: kết quả `7`.

**Bài 2.**
- Mức 1: sắp theo tầng: `0, 4, 9`. Đầu trái là tầng 0, đầu phải là tầng 9.
- Mức 2: đi `4 → 0 → 9` tốn `4 + 9 = 13` giây (hoặc `4 → 9 → 0` tốn `5 + 9 = 14`); quay về tầng 4 thêm 5 giây, tức lúc 18. Yêu cầu tầng 4 chỉ phục vụ được từ lúc 20.
- Mức 3: thang về tầng 4 lúc 18 nên phải chờ tới 20 ⇒ đáp án `20`.

**Bài 3.**
- Mức 1: dùng `abs`.
- Mức 2: `t + abs(pos - floor)` là thời điểm đến.
- Mức 3: `return max(t + abs(pos - floor), arrival)`.

**Bài 4.**
- Mức 1: sắp `requests` theo tầng.
- Mức 2: trạng thái `(l, r, side)`, vị trí `F[l-1]` hoặc `F[m-r]`.
- Mức 3: khởi đầu từ `start`; kết thúc khi `l + r == m`.

**Bài 5.**
- Mức 1: chỉ khác khi kết thúc.
- Mức 2: cộng khoảng cách từ tầng cuối về `start`.
- Mức 3: `best = min(best, t + abs(pos - start))`.

## 19. Đáp án tham khảo

In [19]:
# Bài 1, 2
assert finish_time(2, [(0, 5), (4, 1)]) == 7
assert Solution().elevatorRequests(10, 4, [[0, 0], [0, 9], [20, 4]]) == 20

# Bài 3
def step_ans(t, pos, floor, arrival):
    return max(t + abs(pos - floor), arrival)
assert step_ans(0, 0, 5, 6) == 6 and step_ans(6, 5, 8, 0) == 9
print("Bài 1–3 OK")

Bài 1–3 OK


In [20]:
# Bài 4
def my_elevator_ans(n, start, requests):
    R = sorted((f, a) for a, f in requests)
    m = len(R)
    F = [x[0] for x in R]; A = [x[1] for x in R]
    INF = float("inf")
    dp = [[[INF, INF] for _ in range(m + 1)] for _ in range(m + 1)]
    dp[1][0][0] = max(abs(start - F[0]), A[0])
    dp[0][1][1] = min(dp[0][1][1], max(abs(start - F[m - 1]), A[m - 1]))
    best = INF
    for d in range(1, m + 1):
        for l in range(d + 1):
            r = d - l
            for s in (0, 1):
                t = dp[l][r][s]
                if t == INF:
                    continue
                pos = F[l - 1] if s == 0 else F[m - r]
                if l + r == m:
                    best = min(best, t)
                    continue
                dp[l + 1][r][0] = min(dp[l + 1][r][0], max(t + abs(pos - F[l]), A[l]))
                dp[l][r + 1][1] = min(dp[l][r + 1][1], max(t + abs(pos - F[m - 1 - r]), A[m - 1 - r]))
    return best

# Bài 5
def elevator_round_trip_ans(n, start, requests):
    R = sorted((f, a) for a, f in requests)
    m = len(R)
    F = [x[0] for x in R]; A = [x[1] for x in R]
    INF = float("inf")
    dp = [[[INF, INF] for _ in range(m + 1)] for _ in range(m + 1)]
    dp[1][0][0] = max(abs(start - F[0]), A[0])
    dp[0][1][1] = min(dp[0][1][1], max(abs(start - F[m - 1]), A[m - 1]))
    best = INF
    for d in range(1, m + 1):
        for l in range(d + 1):
            r = d - l
            for s in (0, 1):
                t = dp[l][r][s]
                if t == INF:
                    continue
                pos = F[l - 1] if s == 0 else F[m - r]
                if l + r == m:
                    best = min(best, t + abs(pos - start))
                    continue
                dp[l + 1][r][0] = min(dp[l + 1][r][0], max(t + abs(pos - F[l]), A[l]))
                dp[l][r + 1][1] = min(dp[l][r + 1][1], max(t + abs(pos - F[m - 1 - r]), A[m - 1 - r]))
    return best

rng = random.Random(99)
for _ in range(300):
    m = rng.randint(1, 4)
    n = rng.randint(1, 8)
    reqs = [[rng.randint(0, 8), rng.randint(0, n - 1)] for _ in range(m)]
    s = rng.randint(0, n - 1)
    assert my_elevator_ans(n, s, reqs) == Solution().elevatorRequests(n, s, reqs)
    assert elevator_round_trip_ans(n, s, reqs) == elevator_bfs(n, s, reqs, return_home=True), (n, s, reqs)
assert elevator_round_trip_ans(9, 0, [[0, 8], [6, 5]]) == 16      # đi 8 trước (lúc 8), rồi tầng 5 (lúc 11), về tầng 0: 11 + 5
print("Bài 4–5 OK (bài 5 đối chiếu BFS yêu cầu thang về start)")

Bài 4–5 OK (bài 5 đối chiếu BFS yêu cầu thang về start)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP đoạn:**

```python
R = sorted((floor, arrival) for arrival, floor in requests)   # theo tầng
dp[l][r][side]   # l bóc từ trái, r bóc từ phải; side 0: ở F[l-1]; side 1: ở F[m-r]
dp[1][0][0] = max(abs(start - F[0]), A[0])
dp[0][1][1] = max(abs(start - F[-1]), A[-1])
bóc trái : nt = max(t + abs(pos - F[l]),   A[l])
bóc phải : nt = max(t + abs(pos - F[m-1-r]), A[m-1-r])
đáp án = min dp[l][r][*] với l + r == m
```

**Checklist trước khi submit:**

- [ ] Thời điểm phục vụ = `max(đến nơi, arrival)` (có chờ).
- [ ] Sắp theo **tầng**, không phải theo arrival.
- [ ] Hai trạng thái đầu từ `start`: bóc trái đầu và bóc phải đầu.
- [ ] Vị trí hiện tại: `F[l-1]` (side 0) hoặc `F[m-r]` (side 1).
- [ ] Dừng khi `l + r == m` và lấy `min`.
- [ ] Xử lý yêu cầu trùng tầng (không giả sử tầng khác nhau).
- [ ] Đối chiếu bitmask/BFS trên input nhỏ ngẫu nhiên có seed.